## Prediction of Customer Churn Pbobability

**Version 1.0**

The notebook in fourth entry to competition I started with Basics to start with using Random Fore7r, before I test pipelione and ensemble

**Version 2.0**

Random Forests tend to struggle with this because they average predictions from many trees, which often pushes probabilities away from 0 and 1 toward the center.
This version introduces Random Forest Calibration By applying Platt Scaling (Sigmoid calibration), turns a mathematical output into a reliable financial metric for churn prevention strategies.

**Version 3.0**

Introduced GridSearchCV: It no longer guesses the number of trees; it tests 50, 100, 150, 200, 250 and 300 to find the most accurate estimator for specific data.

**Version 4.0**

Introduced notebook pipeline using Balanced Random Forest (from imblearn) designed to:
- Handle churn imbalance properly
- Increase churn probability sensitivity
- Optimize for ROC-AUC
- Improve recall for churners
- Provide calibrated probabilities
- Use stratified CV
- Automatically tune threshold

**Version 5.0***

Added multi-seed ensemble combining XGBoost + Random Forest, optimized for churn imbalance and probability separation.
- XGBoost with scale_pos_weight
- RandomForest with class_weight='balanced'
- 3 seeds
- 5-fold stratified CV
- Rank normalization blending
- Weight optimization
- Threshold tuning for recall

**Version 6.0**

Added CatBoost to XGBoost and RF
  3-model multi-seed ensemble:
- XGBoost
- Random Forest
- CatBoost
- 3 seeds
- 5-fold stratified CV
- Rank blending
- Weight optimization
- Threshold tuning

**Version 7.0**


Added LightGBM to the previous XGBoost + Random Forest + CatBoost multi-seed ensemble to evaluate 4 model ensemble, these four specific algorithms leverage different mathematical "perspectives" on your data:
- XGBoost & LightGBM: These are "Gradient Boosted Decision Trees" (GBDT). They build trees sequentially, where each tree tries to correct the errors of the previous one. They are highly aggressive and excellent at finding complex patterns.
- CatBoost: It uses a specialized technique called "Ordered Boosting" which is specifically designed to reduce bias when dealing with categorical variables. It often finds insights that XGB/LGB miss.
- Random Forest: This is a "Bagging" model. Unlike the others, it builds many independent trees in parallel and averages them. It is much harder to overfit, making it a perfect "anchor" for your ensemble to prevent the boosting models from becoming too volatile.

**Version 8.0**

Added Feature Engineering along with automatic ensemble weight calculation, the model now has
- Feature Engineering (tenure ratios, charge ratios, service counts)
- Automatic Ensemble Weight Search
- Multi-seed ensemble (XGB + LGBM + CatBoost + RF)
- Rank blending
- Grid search for optimal ensemble weights

In [1]:
# ============================================================
# 1. Imports
# ============================================================

import numpy as np
import pandas as pd
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier

from scipy.stats import rankdata

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

SEEDS = [11, 42, 333, 777]
N_FOLDS = 5
TARGET = "Churn"

# Set visual style
sns.set(style="whitegrid")


In [2]:
# =============================================================================
# 2. Data Loading 
# =============================================================================
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/test.csv")

print("Completed Loading Data ...")
print("Train shape:", train.shape)
print("Test shape :", test.shape)

test_ids = test["id"].copy()

# Descriptive Statistics
stats = train.describe()

print("\nDescriptive Stats:\n", stats)


Completed Loading Data ...
Train shape: (594194, 21)
Test shape : (254655, 20)

Descriptive Stats:
                   id  SeniorCitizen         tenure  MonthlyCharges  \
count  594194.000000  594194.000000  594194.000000   594194.000000   
mean   297096.500000       0.114102      36.577258       65.866223   
std    171529.177262       0.317936      25.061922       31.067444   
min         0.000000       0.000000       1.000000       18.250000   
25%    148548.250000       0.000000      12.000000       29.900000   
50%    297096.500000       0.000000      35.000000       74.100000   
75%    445644.750000       0.000000      62.000000       90.800000   
max    594193.000000       1.000000      72.000000      118.750000   

        TotalCharges  
count  594194.000000  
mean     2494.377057  
std      2353.916710  
min        18.800000  
25%       639.650000  
50%      1433.650000  
75%      4263.800000  
max      8684.800000  


In [3]:
# ============================================================
# 3. Feature Engineering
# ============================================================

def create_features(df):

    df["avg_monthly_charge"] = df["TotalCharges"] / (df["tenure"] + 1)

    df["charge_tenure_ratio"] = df["MonthlyCharges"] / (df["tenure"] + 1)

    df["totalcharge_monthly_ratio"] = df["TotalCharges"] / (df["MonthlyCharges"] + 1)

    service_cols = [
        'PhoneService','InternetService','OnlineSecurity',
        'OnlineBackup','DeviceProtection','TechSupport',
        'StreamingTV','StreamingMovies'
    ]

    df["num_services"] = df[service_cols].astype(str).apply(
        lambda x: (x!="No").sum(),
        axis=1
    )

    return df


train = create_features(train)
test  = create_features(test)

In [4]:
# ============================================================
# 4. Cleaning and Encoding
# ============================================================
for df in [train, test]:

    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

    df["TotalCharges"].fillna(
        df["TotalCharges"].median(),
        inplace=True
    )

cat_cols = train.select_dtypes(include="object").columns.tolist()
cat_cols.remove(TARGET)

encoders = {}

for col in cat_cols:

    le = LabelEncoder()

    train[col] = le.fit_transform(train[col].astype(str))
    test[col]  = le.transform(test[col].astype(str))

    encoders[col] = le


train[TARGET] = train[TARGET].map({"Yes":1,"No":0})

X = train.drop(["id", TARGET], axis=1)
y = train[TARGET]

X_test = test.drop(["id"], axis=1)

print("Churn Rate:", y.mean())

neg = (y == 0).sum()
pos = (y == 1).sum()

scale_pos_weight = neg / pos

print("scale_pos_weight:", scale_pos_weight)

Churn Rate: 0.225207592133209
scale_pos_weight: 3.440347638939746


In [5]:
# ============================================================
# 4. Multi-Seed Training - XGB + RF + CatBoost + LGBM
# ============================================================
all_oof_xgb = []
all_oof_lgb = []
all_oof_rf  = []
all_oof_cat = []

all_test_xgb = []
all_test_lgb = []
all_test_rf  = []
all_test_cat = []

# ============================================================
# 4. Multi-Seed Training - XGB + RF + CatBoost + LGBM
# ============================================================

for seed in SEEDS:

    print("\n" + "="*60)
    print(f"TRAINING WITH SEED {seed}")
    print("="*60)

    skf = StratifiedKFold(
        n_splits=N_FOLDS,
        shuffle=True,
        random_state=seed
    )

    oof_xgb = np.zeros(len(X))
    oof_lgb = np.zeros(len(X))
    oof_rf  = np.zeros(len(X))
    oof_cat = np.zeros(len(X))

    test_xgb = np.zeros(len(X_test))
    test_lgb = np.zeros(len(X_test))
    test_rf  = np.zeros(len(X_test))
    test_cat = np.zeros(len(X_test))

    for fold,(tr_idx,val_idx) in enumerate(skf.split(X,y)):

        print(f"\n----- Fold {fold+1}/{N_FOLDS} -----")

        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]


        # ================= XGBOOST =================

        xgb_model = xgb.XGBClassifier(
            n_estimators=10000,
            learning_rate=0.0125,
            max_depth=6,
            min_child_weight=5,
            subsample=0.8,
            colsample_bytree=0.8,
            gamma=0.1,
            reg_alpha=0.1,
            reg_lambda=1,
            scale_pos_weight=scale_pos_weight,
            tree_method="hist",
            eval_metric="auc",
            random_state=seed,
            n_jobs=-1
        )

        xgb_model.fit(
            X_tr,y_tr,
            eval_set=[(X_val,y_val)],
            verbose=False
        )

        xgb_pred = xgb_model.predict_proba(X_val)[:,1]
        oof_xgb[val_idx] = xgb_pred

        print(f"XGB Fold AUC: {roc_auc_score(y_val,xgb_pred):.5f}")

        test_xgb += xgb_model.predict_proba(X_test)[:,1] / N_FOLDS


        # ================= LIGHTGBM =================

        lgb_model = lgb.LGBMClassifier(
            n_estimators=10000,
            learning_rate=0.0125,
            num_leaves=64,
            subsample=0.8,
            colsample_bytree=0.8,
            class_weight="balanced",
            random_state=seed,
            verbosity=-1,
            force_row_wise=True
        )

        lgb_model.fit(
            X_tr,y_tr,
            eval_set=[(X_val,y_val)],
            eval_metric="auc",
            callbacks=[lgb.early_stopping(200)]
        )

        lgb_pred = lgb_model.predict_proba(X_val)[:,1]
        oof_lgb[val_idx] = lgb_pred

        print(f"LGB Fold AUC: {roc_auc_score(y_val,lgb_pred):.5f}")

        test_lgb += lgb_model.predict_proba(X_test)[:,1] / N_FOLDS


        # ================= RANDOM FOREST =================

        rf = RandomForestClassifier(
            n_estimators=1000,
            min_samples_leaf=3,
            class_weight="balanced",
            random_state=seed,
            n_jobs=-1
        )

        rf.fit(X_tr,y_tr)

        rf_pred = rf.predict_proba(X_val)[:,1]
        oof_rf[val_idx] = rf_pred

        print(f"RF Fold AUC : {roc_auc_score(y_val,rf_pred):.5f}")

        test_rf += rf.predict_proba(X_test)[:,1] / N_FOLDS


        # ================= CATBOOST =================

        cat = CatBoostClassifier(
            iterations=10000,
            learning_rate=0.0125,
            depth=6,
            eval_metric="AUC",
            auto_class_weights="Balanced",
            random_seed=seed,
            verbose=False
        )

        cat.fit(
            X_tr,y_tr,
            eval_set=(X_val,y_val),
            early_stopping_rounds=200
        )

        cat_pred = cat.predict_proba(X_val)[:,1]
        oof_cat[val_idx] = cat_pred

        print(f"CAT Fold AUC: {roc_auc_score(y_val,cat_pred):.5f}")

        test_cat += cat.predict_proba(X_test)[:,1] / N_FOLDS


    print("\nSeed Summary")
    print("---------------------------")

    print("XGB AUC:", roc_auc_score(y,oof_xgb))
    print("LGB AUC:", roc_auc_score(y,oof_lgb))
    print("RF  AUC:", roc_auc_score(y,oof_rf))
    print("CAT AUC:", roc_auc_score(y,oof_cat))


    # Store results
    all_oof_xgb.append(oof_xgb)
    all_oof_lgb.append(oof_lgb)
    all_oof_rf.append(oof_rf)
    all_oof_cat.append(oof_cat)

    all_test_xgb.append(test_xgb)
    all_test_lgb.append(test_lgb)
    all_test_rf.append(test_rf)
    all_test_cat.append(test_cat)


TRAINING WITH SEED 11

----- Fold 1/5 -----
XGB Fold AUC: 0.91385
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[2411]	valid_0's auc: 0.914802	valid_0's binary_logloss: 0.370136
LGB Fold AUC: 0.91480
RF Fold AUC : 0.91023
CAT Fold AUC: 0.91529

----- Fold 2/5 -----
XGB Fold AUC: 0.91521
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[1875]	valid_0's auc: 0.915791	valid_0's binary_logloss: 0.370122
LGB Fold AUC: 0.91579
RF Fold AUC : 0.91068
CAT Fold AUC: 0.91615

----- Fold 3/5 -----
XGB Fold AUC: 0.91605
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[2725]	valid_0's auc: 0.916753	valid_0's binary_logloss: 0.368734
LGB Fold AUC: 0.91675
RF Fold AUC : 0.91213
CAT Fold AUC: 0.91710

----- Fold 4/5 -----
XGB Fold AUC: 0.91637
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[2346]	valid_0's 

In [6]:
# ============================================================
# 5. Average Prediction
# ============================================================
oof_xgb = np.mean(all_oof_xgb, axis=0)
oof_lgb = np.mean(all_oof_lgb, axis=0)
oof_rf  = np.mean(all_oof_rf, axis=0)
oof_cat = np.mean(all_oof_cat, axis=0)

test_xgb = np.mean(all_test_xgb, axis=0)
test_lgb = np.mean(all_test_lgb, axis=0)
test_rf  = np.mean(all_test_rf, axis=0)
test_cat = np.mean(all_test_cat, axis=0)

print("XGB AUC:", roc_auc_score(y,oof_xgb))
print("LGB AUC:", roc_auc_score(y,oof_lgb))
print("RF  AUC:", roc_auc_score(y,oof_rf))
print("CAT AUC:", roc_auc_score(y,oof_cat))

XGB AUC: 0.9159117939321532
LGB AUC: 0.9163871052189919
RF  AUC: 0.9122929392102834
CAT AUC: 0.9166892404854091


In [7]:
# ============================================================
# 6. Rank Normalization
# ============================================================
def rank_norm(x):
    return rankdata(x) / len(x)

oof_xgb_r = rank_norm(oof_xgb)
oof_lgb_r = rank_norm(oof_lgb)
oof_rf_r  = rank_norm(oof_rf)
oof_cat_r = rank_norm(oof_cat)

test_xgb_r = rank_norm(test_xgb)
test_lgb_r = rank_norm(test_lgb)
test_rf_r  = rank_norm(test_rf)
test_cat_r = rank_norm(test_cat)

In [8]:
# ============================================================
# 7. Automatic Ensemble Weight Search
# ============================================================

best_auc = 0
best_weights = None

for wx in np.arange(0.1,0.6,0.1):
    for wl in np.arange(0.1,0.6,0.1):
        for wc in np.arange(0.1,0.6,0.1):

            wr = 1 - wx - wl - wc

            if wr <= 0:
                continue

            blend = (
                wx*oof_xgb_r +
                wl*oof_lgb_r +
                wc*oof_cat_r +
                wr*oof_rf_r
            )

            auc = roc_auc_score(y, blend)

            if auc > best_auc:

                best_auc = auc
                best_weights = (wx, wl, wc, wr)


print("\nBest Ensemble Weights")
print("----------------------")

print("XGB :", best_weights[0])
print("LGB :", best_weights[1])
print("CAT :", best_weights[2])
print("RF  :", best_weights[3])

print("Best AUC:", best_auc)



Best Ensemble Weights
----------------------
XGB : 0.2
LGB : 0.2
CAT : 0.5
RF  : 0.10000000000000009
Best AUC: 0.9167653596462579


In [9]:
# =============================================================================
# 7. Final Prediction and Submission
# =============================================================================
wx, wl, wc, wr = best_weights

final_test = (
    wx*test_xgb_r +
    wl*test_lgb_r +
    wc*test_cat_r +
    wr*test_rf_r
)

submission = pd.DataFrame({
    "id": test_ids,
    "Churn_Probability": final_test
})

submission.to_csv("submission.csv", index=False)

submission.head(20)




,id,Churn_Probability
0,594194,0.511856
1,594195,0.025252
2,594196,0.552686
3,594197,0.155070
4,594198,0.797713
5,594199,0.636741
6,594200,0.980900
7,594201,0.156862
8,594202,0.427071
9,594203,0.712429
